In [ ]:
import pandas as pd


def find_mutual_selections(data):
    # 标准化输入
    records = (
        data.to_dict('records') if isinstance(data, pd.DataFrame) else data
    )

    males, females = {}, {}
    for r in records:
        gender = str(r['性别']).strip()
        num = str(r['号牌编号']).strip()
        choices = [
            c.strip() for c in str(r['心动异性']).split(',') if c.strip()
        ]
        if gender == '男':
            males[num] = choices
        else:
            females[num] = choices

    # 计算互选列表
    mutual_male = {
        m: [f for f in choices if f in females and m in females[f]]
        for m, choices in males.items()
    }
    mutual_female = {
        f: [m for m in choices if m in males and f in males[m]]
        for f, choices in females.items()
    }

    one_to_one, one_to_many_dict = [], {}

    # 遍历所有互选配对（以男性为主体避免重复）
    for m, female_partners in mutual_male.items():
        if not female_partners:
            continue

        male_mutual_count = len(female_partners)

        for f in female_partners:
            female_mutual_count = len(mutual_female[f])

            # 只有双方都是1对1才归入一对一
            if male_mutual_count == 1 and female_mutual_count == 1:
                one_to_one.append(
                    {
                        '男号': m,
                        '女号': f,
                        '男的互选人数': male_mutual_count,
                        '女的互选人数': female_mutual_count,
                    }
                )
            else:
                # 任何一方是一对多就归入一对多，按男号分组
                if m not in one_to_many_dict:
                    one_to_many_dict[m] = []
                one_to_many_dict[m].append(f)

    # 将一对多字典转换为DataFrame格式
    one_to_many = [
        {
            '男号': m,
            '女号': ','.join(females_list),
            '互选人数': len(females_list),
        }
        for m, females_list in one_to_many_dict.items()
    ]

    # 统计被选择次数（人气王）
    male_choice_count = {}  # 男的被选择次数
    female_choice_count = {}  # 女的被选择次数

    for m, choices in males.items():
        for f in choices:
            female_choice_count[f] = female_choice_count.get(f, 0) + 1

    for f, choices in females.items():
        for m in choices:
            male_choice_count[m] = male_choice_count.get(m, 0) + 1

    # 找出人气王
    male_king = (
        max(male_choice_count.items(), key=lambda x: x[1])
        if male_choice_count
        else (None, 0)
    )
    female_king = (
        max(female_choice_count.items(), key=lambda x: x[1])
        if female_choice_count
        else (None, 0)
    )

    return {
        'one_to_one': pd.DataFrame(one_to_one),
        'one_to_many': pd.DataFrame(one_to_many),
        'popularity': {
            '男人气王': {'号码': male_king[0], '被选次数': male_king[1]},
            '女人气王': {'号码': female_king[0], '被选次数': female_king[1]},
        },
        'summary': {
            '一对一互选': len(one_to_one),
            '一对多互选组数': len(one_to_many),
            '一对多互选总对数': sum(
                [len(v) for v in one_to_many_dict.values()]
            ),
        },
    }


# 使用示例（文件读取）
df = pd.read_excel(
    r'C:\Users\admin\Documents\xwechat_files\wxid_otuh0yh5qwvp22_7de6\msg\file\2026-01\0125招行&教育系统单身青年交友联谊活动心仪表.xlsx'
)
data = df[
    ["你的性别（必填）", "你的号牌编号（必填）", "你的心动异性（必填）"]
].rename(
    columns={
        "你的性别（必填）": "性别",
        "你的号牌编号（必填）": "号牌编号",
        "你的心动异性（必填）": "心动异性",
    }
)

result = find_mutual_selections(data)

print("一对一互选：")
print(result['one_to_one'])
print("\n一对多互选：")
print(result['one_to_many'])
print("\n人气王统计：")
print(
    f"男人气王：{result['popularity']['男人气王']['号码']} (被选{result['popularity']['男人气王']['被选次数']}次)"
)
print(
    f"女人气王：{result['popularity']['女人气王']['号码']} (被选{result['popularity']['女人气王']['被选次数']}次)"
)
print("\n统计摘要：", result['summary'])

# 保存
with pd.ExcelWriter('互选结果.xlsx', engine='openpyxl') as writer:
    result['one_to_one'].to_excel(writer, sheet_name='一对一互选', index=False)
    result['one_to_many'].to_excel(writer, sheet_name='一对多互选', index=False)

    # 人气王也保存到Excel
    popularity_df = pd.DataFrame(
        [
            {
                '类别': '男人气王',
                '号码': result['popularity']['男人气王']['号码'],
                '被选次数': result['popularity']['男人气王']['被选次数'],
            },
            {
                '类别': '女人气王',
                '号码': result['popularity']['女人气王']['号码'],
                '被选次数': result['popularity']['女人气王']['被选次数'],
            },
        ]
    )
    popularity_df.to_excel(writer, sheet_name='人气王', index=False)
pd.concat(
    [
        pd.DataFrame([{'男号': '一对一', '女号': ''}]),
        result['one_to_one'][['男号', '女号']],
        pd.DataFrame([{'男号': '一对多', '女号': ''}]),
        result['one_to_many'][['男号', '女号']],
        pd.DataFrame([{'男号': '人气王', '女号': ''}]),
        pd.DataFrame(
            [
                {
                    '男号': result['popularity']['男人气王']['号码'],
                    '女号': result['popularity']['女人气王']['号码'],
                }
            ]
        ),
    ]
).to_excel('互选结果_合并版.xlsx', index=False, header=None)